# LAB 2 — Extraction of Latent Features using CNN for Image Datasets
**Name:** Shivangi Singh  **Department:** Computer Science

### AIM
To understand the process of extracting features from images using Convolutional Neural Networks (CNN) and apply CNN models for image classification by analysing the effect of different hyperparameters.

### What this notebook covers (mapped to the lab sheet)

| Lab requirement | Section |
|---|---|
| Grayscale medical image dataset (chest X-ray) | A1 |
| Original grayscale medical image | A1 |
| Resize, normalise, convert to tensor | A2 |
| Filters — **user-defined** (hand-written convolution + hand-designed kernels) and **in-built** (`F.conv2d`, `nn.Conv2d`, `skimage`) | A3 |
| CNN architecture: Conv → ReLU → Pool → Flatten → Dense (latent) | A4 |
| Intermediate feature maps after each convolution layer | A6 |
| Feature maps after pooling layers / after every CNN layer | A6 |
| Final latent feature vector representation | A6, A7 |
| Image classification + effect of hyperparameters | A5, A8 |
| RGB image + preprocessing | B1, B2 |
| Edge / texture / shape / high-level semantic features | B3, B5, B6 |
| Output channels after each conv layer, pooling outputs, flattened features, latent vector before classifier | B4, B5 |
| Comparison of how CNN extracts different levels of features | B6, C |

### Complexity elements added
1. **From-scratch implementations** of convolution (naive loop + vectorised), multi-channel convolution, ReLU and pooling, numerically verified against PyTorch's in-built ops, with runtime comparison.
2. **Hybrid first conv layer**: fixed user-defined filters (Sobel, Prewitt, Laplacian, Gaussian, …) concatenated with learnable in-built `nn.Conv2d` filters inside the trained CNN.
3. **Computational complexity analysis**: output-size formula, parameters, MACs (multiply–accumulate operations) and receptive field per layer.
4. **Training + evaluation** on a real medical dataset with class-imbalance handling (weighted loss) and multiple metrics (Accuracy, ROC-AUC, F1, confusion matrix).
5. **Systematic hyperparameter study** (learning rate, kernel size, number of filters, optimiser, batch size, pooling type, activation, dropout, latent dimension).
6. **Latent-space analysis**: PCA, t-SNE, and a *linear probe* comparing raw pixels vs untrained-CNN latents vs trained-CNN latents.
7. **Explainability**: Grad-CAM for both the medical CNN and the RGB CNN.
8. **Feature-hierarchy quantification** for RGB: receptive field, sparsity, effective rank, invariance to perturbations, and cross-image semantic similarity per layer.

> ⚠️ Runs on Google Colab or locally (CUDA / Apple-Silicon MPS / CPU). Needs internet on first run (the dataset and pretrained weights are downloaded on first run). Set `QUICK_RUN = True` in the config cell for a fast (lower-accuracy) run.

## 0. Setup

In [2]:
# # Check / install required packages. medmnist is NOT needed (the data file is downloaded directly).
# import importlib.util, subprocess, sys
# REQUIRED = {"torch": "torch", "torchvision": "torchvision", "numpy": "numpy", "pandas": "pandas",
#             "matplotlib": "matplotlib", "sklearn": "scikit-learn", "skimage": "scikit-image", "scipy": "scipy"}
# missing = [pip_name for mod, pip_name in REQUIRED.items() if importlib.util.find_spec(mod) is None]
# print("Python", sys.version.split()[0], "| interpreter:", sys.executable)
# if missing:
#     print("Installing:", missing)
#     r = subprocess.run([sys.executable, "-m", "pip", "install", *missing], capture_output=True, text=True)
#     if r.returncode != 0:
#         print(r.stdout[-3000:], r.stderr[-3000:])
#         print("\n❌ pip failed (real error shown above). Fix it in a terminal, then restart the kernel.")
#     else:
#         print("✅ Installed. Restart the kernel if an import below still fails.")
# else:
#     print("✅ All required packages present.")

In [3]:
import os, time, copy, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from skimage import data as skdata, feature, filters, measure, morphology, color, draw, io as skio
from skimage import transform as sktransform
from numpy.lib.stride_tricks import sliding_window_view

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 100

# ------------------------- CONFIG -------------------------
SEED        = 42
QUICK_RUN   = False          # True -> fewer epochs / smaller subsets (for a fast demo)
RAW_SIZE    = 128            # resolution downloaded from MedMNIST (64 / 128 / 224 available)
IMG_SIZE    = 64             # resize target for the medical CNN
LATENT_DIM  = 128            # size of latent vector (dense layer)
EPOCHS      = 3 if QUICK_RUN else 10     # epochs for the main medical CNN
HP_EPOCHS   = 1 if QUICK_RUN else 4      # epochs per hyperparameter-study run
HP_TRAIN_SUBSET = 800 if QUICK_RUN else 2500   # training images per HP run (keeps the study tractable)
HP_SEEDS    = [42]           # add more seeds, e.g. [42, 7, 123], for error bars (slower but more reliable)
DATA_DIR    = "./data"
RGB_IMAGE_PATH = None        # e.g. "my_dog.jpg"; None -> scikit-image's built-in cat photo
RGB_SIZE    = 224
# ----------------------------------------------------------

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

ModuleNotFoundError: No module named 'torch'

## 1. Utility functions — user-defined (from-scratch) CNN operations

CNN "convolution" layers actually compute **cross-correlation**:

$$Y(i,j)=\sum_{m=0}^{k-1}\sum_{n=0}^{k-1} X(i\cdot s+m,\; j\cdot s+n)\,K(m,n) + b$$

Output size: $\;O = \left\lfloor \dfrac{N - k + 2p}{s} \right\rfloor + 1$ where $N$ = input size, $k$ = kernel size, $p$ = padding, $s$ = stride.

Cost of one conv layer: $\text{MACs} = H_{out}\cdot W_{out}\cdot C_{out}\cdot C_{in}\cdot k^2$.

In [ ]:
# ---------------- user-defined operations (NumPy, no deep-learning library) ----------------
def conv2d_manual(img, kernel, stride=1, padding=0):
    """Naive 2-D cross-correlation with explicit loops (single channel)."""
    img = np.asarray(img, dtype=np.float64); k = np.asarray(kernel, dtype=np.float64)
    if padding > 0:
        img = np.pad(img, padding, mode="constant")
    kh, kw = k.shape; H, W = img.shape
    oh, ow = (H - kh) // stride + 1, (W - kw) // stride + 1
    out = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            r, c = i * stride, j * stride
            out[i, j] = np.sum(img[r:r + kh, c:c + kw] * k)
    return out

def conv2d_manual_vec(img, kernel, stride=1, padding=0):
    """Vectorised version of the same operation using sliding windows."""
    img = np.asarray(img, dtype=np.float64); k = np.asarray(kernel, dtype=np.float64)
    if padding > 0:
        img = np.pad(img, padding, mode="constant")
    win = sliding_window_view(img, k.shape)[::stride, ::stride]
    return np.einsum("ijkl,kl->ij", win, k)

def conv2d_manual_multichannel(img_chw, kernel_chw, bias=0.0, stride=1, padding=0):
    """Multi-channel conv: one 3-D kernel (C x k x k) -> one output channel (sum over input channels)."""
    return sum(conv2d_manual_vec(img_chw[c], kernel_chw[c], stride, padding)
               for c in range(img_chw.shape[0])) + bias

def relu_manual(x):
    return np.maximum(0, x)

def pool2d_manual(x, size=2, stride=2, mode="max"):
    win = sliding_window_view(np.asarray(x, dtype=np.float64), (size, size))[::stride, ::stride]
    return win.max(axis=(-1, -2)) if mode == "max" else win.mean(axis=(-1, -2))

def out_size(n, k, s=1, p=0):
    return (n - k + 2 * p) // s + 1

# ---------------- user-defined filter kernels ----------------
def gaussian_kernel(size=3, sigma=1.0):
    ax = np.arange(size) - (size - 1) / 2
    xx, yy = np.meshgrid(ax, ax)
    g = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    return g / g.sum()

def gabor_kernel_manual(size=15, sigma=3.0, theta=0.0, lambd=8.0, gamma=0.5, psi=0.0):
    """Real part of a Gabor filter written from its formula (texture detector)."""
    half = size // 2
    y, x = np.mgrid[-half:half + 1, -half:half + 1].astype(np.float64)
    xr = x * np.cos(theta) + y * np.sin(theta)
    yr = -x * np.sin(theta) + y * np.cos(theta)
    g = np.exp(-(xr**2 + (gamma**2) * yr**2) / (2 * sigma**2)) * np.cos(2 * np.pi * xr / lambd + psi)
    return g - g.mean()          # zero-mean -> no response to flat regions

USER_FILTERS = {
    "Sobel-X":   np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float),
    "Sobel-Y":   np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], float),
    "Prewitt-X": np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], float),
    "Prewitt-Y": np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], float),
    "Laplacian": np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], float),
    "Gaussian":  gaussian_kernel(3, 1.0),
    "Sharpen":   np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], float),
    "Emboss":    np.array([[-2, -1, 0], [-1, 1, 1], [0, 1, 2]], float),
}

# ---------------- plotting helpers ----------------
def show_images(imgs, titles=None, ncols=4, cmap="gray", size=3, suptitle=None, colorbar=False):
    n = len(imgs); nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(size * ncols, size * nrows), squeeze=False)
    for ax in axes.ravel(): ax.axis("off")
    for i, im in enumerate(imgs):
        ax = axes.ravel()[i]
        h = ax.imshow(im, cmap=cmap if np.ndim(im) == 2 else None)
        if titles: ax.set_title(titles[i], fontsize=9)
        if colorbar and np.ndim(im) == 2: plt.colorbar(h, ax=ax, fraction=0.046)
    if suptitle: fig.suptitle(suptitle, fontsize=12, fontweight="bold")
    plt.tight_layout(); plt.show()

def show_feature_maps(fmap, title, max_maps=16, ncols=8, cmap="viridis", names=None):
    """fmap: tensor/array [C, H, W]"""
    fmap = fmap.detach().cpu().numpy() if torch.is_tensor(fmap) else np.asarray(fmap)
    C = fmap.shape[0]; n = min(C, max_maps)
    titles = [(names[i] if names and i < len(names) else f"ch {i}") for i in range(n)]
    show_images([fmap[i] for i in range(n)], titles, ncols=min(ncols, n), cmap=cmap, size=1.8,
                suptitle=f"{title}  —  shape {tuple(fmap.shape)} (showing {n}/{C} channels)")

class FeatureExtractor:
    """Registers forward hooks and stores the output of named layers."""
    def __init__(self, model, layer_names):
        self.model, self.feats, self.handles = model, {}, []
        modules = dict(model.named_modules())
        for name in layer_names:
            self.handles.append(modules[name].register_forward_hook(self._hook(name)))
    def _hook(self, name):
        def fn(m, i, o): self.feats[name] = o.detach().clone().cpu()
        return fn
    def __call__(self, x):
        self.feats = {}
        self.model.eval()
        with torch.no_grad():
            out = self.model(x)
        return self.feats, out
    def remove(self):
        for h in self.handles: h.remove()

def grad_cam(model, x, target_layer, class_idx=None):
    """Grad-CAM: weight the target layer's maps by the spatially-averaged gradient of the class score."""
    store = {}
    def fwd(m, i, o):
        store["a"] = o
        o.register_hook(lambda g: store.__setitem__("g", g))
    h = target_layer.register_forward_hook(fwd)
    model.eval(); model.zero_grad()
    out = model(x)
    if class_idx is None: class_idx = int(out.argmax(1).item())
    out[0, class_idx].backward()
    h.remove()
    w = store["g"].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * store["a"]).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam.detach().cpu().numpy(), class_idx, out.softmax(1)[0, class_idx].item()

print("Utilities ready. User-defined filters:", list(USER_FILTERS))

---
# PART A — Grayscale Medical Image: Latent Feature Extraction with CNN

## A1. Dataset: PneumoniaMNIST (chest X-ray)
**PneumoniaMNIST** (from the MedMNIST benchmark) contains pediatric chest X-ray images labelled *Normal* vs *Pneumonia* (grayscale, single channel). Official splits: train / validation / test. The dataset is imbalanced (more Pneumonia than Normal), which we handle in training.

The `.npz` file is downloaded directly from the official Zenodo record (no `medmnist` package required). If the download fails, the notebook prints the URL for a manual download and falls back to a **synthetic CT-phantom dataset** so the code still runs — results from the fallback are **not medically meaningful**.

In [ ]:
def make_synthetic_split(n, size, rng):
    """Fallback only: Shepp-Logan CT phantom + noise; class 1 has extra blurred 'opacities'."""
    base = sktransform.resize(skdata.shepp_logan_phantom(), (size, size), anti_aliasing=True)
    X, y = [], []
    for i in range(n):
        lab = i % 2
        img = base * rng.uniform(0.7, 1.1) + rng.normal(0, 0.05, (size, size))
        if lab:
            for _ in range(rng.integers(1, 4)):
                rr, cc = draw.ellipse(rng.integers(size // 4, 3 * size // 4), rng.integers(size // 4, 3 * size // 4),
                                      rng.integers(size // 16, size // 6), rng.integers(size // 16, size // 6), shape=(size, size))
                img[rr, cc] += rng.uniform(0.2, 0.4)
        img = filters.gaussian(img, sigma=1)
        X.append((np.clip(img, 0, 1) * 255).astype(np.uint8)); y.append(lab)
    return np.stack(X), np.array(y, dtype=np.int64)

def load_medical_dataset(size=RAW_SIZE):
    """Downloads PneumoniaMNIST (MedMNIST+) .npz straight from the official Zenodo record (no medmnist package needed).
    If the download fails, manually download the file from the URL printed below and put it in DATA_DIR."""
    import urllib.request, ssl
    fname = f"pneumoniamnist_{size}.npz"
    url = f"https://zenodo.org/records/10519652/files/{fname}?download=1"
    path = os.path.join(DATA_DIR, fname)
    try:
        os.makedirs(DATA_DIR, exist_ok=True)
        if not os.path.exists(path):
            print("Downloading", url)
            try:
                import certifi; ctx = ssl.create_default_context(cafile=certifi.where())
            except ImportError:
                ctx = ssl.create_default_context()
            with urllib.request.urlopen(url, context=ctx, timeout=120) as r, open(path + ".part", "wb") as f:
                f.write(r.read())
            os.replace(path + ".part", path)
        d = np.load(path)
        splits = {s: (d[f"{s}_images"].astype(np.uint8), d[f"{s}_labels"].squeeze().astype(np.int64))
                  for s in ["train", "val", "test"]}
        return splits, "PneumoniaMNIST (chest X-ray)", ["Normal", "Pneumonia"], False
    except Exception as e:
        print("⚠️  PneumoniaMNIST download/load failed ->", repr(e)[:200])
        print(f"⚠️  Manual fix: download {url}\n    and save it as {os.path.abspath(path)} , then re-run this cell.")
        print("⚠️  Using SYNTHETIC fallback data for now. Results are NOT medically meaningful.")
        rng = np.random.default_rng(SEED)
        splits = {s: make_synthetic_split(n, size, rng) for s, n in [("train", 1200), ("val", 200), ("test", 400)]}
        return splits, "Synthetic CT phantom (fallback)", ["Class 0", "Class 1"], True

splits, DATASET_NAME, CLASS_NAMES, IS_FALLBACK = load_medical_dataset()
X_train_raw, y_train = splits["train"]; X_val_raw, y_val = splits["val"]; X_test_raw, y_test = splits["test"]

summary = pd.DataFrame({s: {"images": len(splits[s][1]),
                            **{f"#{c}": int((splits[s][1] == i).sum()) for i, c in enumerate(CLASS_NAMES)}}
                        for s in splits}).T
print("Dataset:", DATASET_NAME, "| raw image shape:", X_train_raw.shape[1:], "| dtype:", X_train_raw.dtype)
display(summary)

In [ ]:
# ---- Original grayscale medical images ----
idx_per_class = [int(np.where(y_train == c)[0][0]) for c in range(len(CLASS_NAMES))]
SAMPLE_IDX = idx_per_class[1]              # sample image used for all visualisations below
sample_raw = X_train_raw[SAMPLE_IDX]

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for k, c in enumerate(idx_per_class):
    axes[k].imshow(X_train_raw[c], cmap="gray"); axes[k].set_title(f"Original — {CLASS_NAMES[y_train[c]]}"); axes[k].axis("off")
axes[2].imshow(sample_raw, cmap="gray"); axes[2].set_title(f"Selected sample (idx {SAMPLE_IDX})"); axes[2].axis("off")
axes[3].hist(sample_raw.ravel(), bins=64, color="gray"); axes[3].set_title("Pixel-intensity histogram (0–255)")
plt.tight_layout(); plt.show()
print(f"Sample: shape={sample_raw.shape}, min={sample_raw.min()}, max={sample_raw.max()}, mean={sample_raw.mean():.1f}")

## A2. Input image pre-processing
1. **Resize** `RAW_SIZE × RAW_SIZE → IMG_SIZE × IMG_SIZE` (anti-aliased) — reduces computation while preserving lung structure.
2. **Normalise** intensities: min–max scaling to $[0,1]$, then z-score standardisation $x' = (x-\mu)/\sigma$ using the **training-set** mean and std (never test statistics → no data leakage).
3. **Convert to tensor** of shape `[N, C, H, W] = [1, 1, IMG_SIZE, IMG_SIZE]`.

Done twice: once **manually (user-defined, NumPy)** and once with **in-built `torchvision.transforms`**, and compared.

In [ ]:
DS_MEAN = float((X_train_raw / 255.0).mean()); DS_STD = float((X_train_raw / 255.0).std())
print(f"Training-set statistics after /255:  mean={DS_MEAN:.4f}, std={DS_STD:.4f}")

# ---------- (1) user-defined pipeline ----------
img_resized = sktransform.resize(sample_raw, (IMG_SIZE, IMG_SIZE), anti_aliasing=True, preserve_range=True)
img_minmax  = img_resized / 255.0
img_z       = (img_minmax - DS_MEAN) / DS_STD
tensor_manual = torch.from_numpy(img_z).float().unsqueeze(0).unsqueeze(0)      # [1,1,H,W]

# ---------- (2) in-built pipeline ----------
inbuilt_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE), antialias=True),
    transforms.ToTensor(),                         # -> [0,1], shape [C,H,W]
    transforms.Normalize([DS_MEAN], [DS_STD]),
])
tensor_inbuilt = inbuilt_tf(sample_raw).unsqueeze(0)

print("Manual tensor :", tuple(tensor_manual.shape), tensor_manual.dtype)
print("In-built tensor:", tuple(tensor_inbuilt.shape), tensor_inbuilt.dtype)
print("Mean |manual - inbuilt| =", float((tensor_manual - tensor_inbuilt).abs().mean()),
      "(small non-zero difference is expected: skimage and PIL use different resampling kernels)")

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
stages = [(sample_raw, f"Original {sample_raw.shape}"), (img_resized, f"Resized {img_resized.shape}"),
          (img_minmax, "Min-max normalised [0,1]"), (img_z, "Z-score standardised")]
for k, (im, t) in enumerate(stages):
    axes[0, k].imshow(im, cmap="gray"); axes[0, k].set_title(t); axes[0, k].axis("off")
    axes[1, k].hist(np.ravel(im), bins=50, color="steelblue"); axes[1, k].set_title(f"range [{np.min(im):.2f}, {np.max(im):.2f}]")
plt.suptitle("Pre-processing pipeline (top: image, bottom: histogram)", fontweight="bold"); plt.tight_layout(); plt.show()

In [ ]:
# ---------- Pre-process whole dataset (vectorised, in-built ops) ----------
def preprocess_array(X, batch=512):
    out = []
    for i in range(0, len(X), batch):
        t = torch.from_numpy(X[i:i + batch]).float().unsqueeze(1) / 255.0
        t = F.interpolate(t, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", antialias=True, align_corners=False)
        out.append((t - DS_MEAN) / DS_STD)
    return torch.cat(out)

Xtr, Xva, Xte = preprocess_array(X_train_raw), preprocess_array(X_val_raw), preprocess_array(X_test_raw)
ytr, yva, yte = map(torch.from_numpy, (y_train, y_val, y_test))
print("Train tensor:", tuple(Xtr.shape), "| Val:", tuple(Xva.shape), "| Test:", tuple(Xte.shape))
x_sample = Xtr[SAMPLE_IDX:SAMPLE_IDX + 1]          # [1,1,H,W] used for feature visualisation

## A3. Filters: user-defined vs in-built
Three ways of applying filters:
* **User-defined kernels + user-defined convolution** — `conv2d_manual` (explicit loops, written above).
* **User-defined kernels + in-built convolution** — the same kernels passed to `torch.nn.functional.conv2d`.
* **In-built filters** — `skimage.filters.sobel/laplace/gaussian` and a randomly-initialised `nn.Conv2d` layer (the filters a CNN starts from before training).

In [ ]:
x_np = img_minmax                                        # [H,W] in [0,1]
names = list(USER_FILTERS); kernels = np.stack([USER_FILTERS[n] for n in names])

# (1) user-defined convolution
t0 = time.perf_counter(); manual_out = np.stack([conv2d_manual(x_np, k, padding=1) for k in kernels]); t_naive = time.perf_counter() - t0
t0 = time.perf_counter(); vec_out = np.stack([conv2d_manual_vec(x_np, k, padding=1) for k in kernels]); t_vec = time.perf_counter() - t0
# (2) in-built convolution with the same user-defined kernels (all 8 in ONE call)
t_in = torch.from_numpy(x_np).float()[None, None]
w = torch.from_numpy(kernels).float()[:, None]           # [8,1,3,3]
builtin_out = F.conv2d(t_in, w, padding=1)[0].numpy()          # warm-up call (first call has one-time overhead)
t0 = time.perf_counter()
for _ in range(20): F.conv2d(t_in, w, padding=1)
t_torch = (time.perf_counter() - t0) / 20

print(f"max |manual - F.conv2d| = {np.abs(manual_out - builtin_out).max():.2e}   (≈0 -> implementation verified)")
print(f"max |naive - vectorised| = {np.abs(manual_out - vec_out).max():.2e}")
print(f"Runtime for 8 filters on {IMG_SIZE}x{IMG_SIZE}:  naive loops {t_naive*1e3:.1f} ms | vectorised {t_vec*1e3:.1f} ms | F.conv2d {t_torch*1e3:.2f} ms")

# show kernels + responses
show_images(list(kernels), [f"{n} kernel" for n in names], ncols=8, cmap="bwr", size=1.9, suptitle="User-defined 3×3 kernels")
show_images(list(manual_out), [f"{n} (manual conv)" for n in names], ncols=8, size=1.9, suptitle="Responses: user-defined convolution")
show_images(list(builtin_out), [f"{n} (F.conv2d)" for n in names], ncols=8, size=1.9, suptitle="Responses: in-built F.conv2d, same kernels")

In [ ]:
# (3) fully in-built filters
sk = [filters.sobel(x_np), filters.laplace(x_np), filters.gaussian(x_np, sigma=1), filters.unsharp_mask(x_np, radius=1, amount=2)]
show_images(sk, ["skimage.sobel (magnitude)", "skimage.laplace", "skimage.gaussian σ=1", "skimage.unsharp_mask"],
            ncols=4, size=3, suptitle="In-built skimage filters")

set_seed()
rand_conv = nn.Conv2d(1, 8, 3, padding=1)               # PyTorch default (Kaiming-uniform) random filters
with torch.no_grad(): rand_out = rand_conv(t_in)[0]
show_images([rand_conv.weight[i, 0].detach().numpy() for i in range(8)], [f"random k{i}" for i in range(8)],
            ncols=8, cmap="bwr", size=1.9, suptitle="In-built nn.Conv2d random-initialised kernels (before training)")
show_feature_maps(rand_out, "nn.Conv2d (random init) output", max_maps=8, cmap="gray")

# ReLU and pooling: user-defined vs in-built on the Sobel-X response
sob = manual_out[0]
relu_m, relu_b = relu_manual(sob), F.relu(torch.from_numpy(sob)).numpy()
mp_m, mp_b = pool2d_manual(relu_m, 2, 2, "max"), nn.MaxPool2d(2)(torch.from_numpy(relu_m)[None, None])[0, 0].numpy()
ap_m, ap_b = pool2d_manual(relu_m, 2, 2, "avg"), nn.AvgPool2d(2)(torch.from_numpy(relu_m)[None, None])[0, 0].numpy()
print("ReLU diff:", np.abs(relu_m - relu_b).max(), "| MaxPool diff:", np.abs(mp_m - mp_b).max(), "| AvgPool diff:", np.abs(ap_m - ap_b).max())
show_images([sob, relu_m, mp_m, ap_m], [f"Sobel-X conv {sob.shape}", "after ReLU", f"MaxPool 2×2 {mp_m.shape}", f"AvgPool 2×2 {ap_m.shape}"],
            ncols=4, size=3, suptitle="Conv → ReLU → Pool (user-defined, verified against in-built)")

In [ ]:
# Output-size formula check: stride / padding / kernel size
rows = []
for k in [3, 5, 7]:
    for s in [1, 2]:
        for p in [0, k // 2]:
            formula = out_size(IMG_SIZE, k, s, p)
            actual = conv2d_manual_vec(x_np, np.ones((k, k)) / k**2, stride=s, padding=p).shape[0]
            rows.append({"kernel": k, "stride": s, "padding": p, "formula O": formula, "actual": actual,
                         "MACs (1 filter)": formula**2 * k * k})
display(pd.DataFrame(rows))

## A4. CNN architecture for latent feature extraction (MedCNN)

```
Input [1×64×64]
 └─ Conv1 (HybridConv: 8 fixed user-defined filters + learnable nn.Conv2d filters) → ReLU → MaxPool 2×2   → [16×32×32]
 └─ Conv2 (nn.Conv2d 3×3) → ReLU → MaxPool 2×2                                                          → [32×16×16]
 └─ Conv3 (nn.Conv2d 3×3) → ReLU → MaxPool 2×2                                                          → [64×8×8]
 └─ Flatten                                                                                             → [4096]
 └─ Dense (fc_latent) → ReLU   ==> LATENT VECTOR                                                        → [128]
 └─ Dropout → Dense classifier                                                                          → [2]
```
The first layer embeds the **user-defined kernels as fixed (non-trainable) filters** alongside **learnable in-built filters**, so both kinds of filter are used *inside* the network.

In [ ]:
def make_handcrafted_bank(k, n):
    """First n user-defined 3x3 kernels, L1-normalised and zero-padded to k x k."""
    bank = []
    for name in list(USER_FILTERS)[:n]:
        f = USER_FILTERS[name] / (np.abs(USER_FILTERS[name]).sum() + 1e-8)
        bank.append(np.pad(f, (k - 3) // 2))
    return torch.tensor(np.stack(bank)[:, None], dtype=torch.float32)

class HybridConv(nn.Module):
    """Conv layer = [fixed user-defined filters (buffer, not trained)] ⊕ [learnable nn.Conv2d filters]."""
    def __init__(self, in_ch, out_ch, k, use_handcrafted=True):
        super().__init__()
        self.n_fixed = min(len(USER_FILTERS), out_ch // 2) if use_handcrafted else 0
        self.pad = k // 2
        if self.n_fixed:
            bank = make_handcrafted_bank(k, self.n_fixed).repeat(1, in_ch, 1, 1) / in_ch
            self.register_buffer("fixed_w", bank)
        self.learn = nn.Conv2d(in_ch, out_ch - self.n_fixed, k, padding=self.pad)
    def forward(self, x):
        y = self.learn(x)
        if self.n_fixed:
            y = torch.cat([F.conv2d(x, self.fixed_w, padding=self.pad), y], dim=1)
        return y

ACTS = {"relu": nn.ReLU, "leaky_relu": lambda: nn.LeakyReLU(0.1), "tanh": nn.Tanh, "sigmoid": nn.Sigmoid}

class MedCNN(nn.Module):
    def __init__(self, in_ch=1, n_filters=16, kernel_size=3, pool="max", act="relu", dropout=0.3,
                 latent_dim=LATENT_DIM, img_size=IMG_SIZE, n_classes=2, use_handcrafted=True):
        super().__init__()
        Act = ACTS[act]; Pool = nn.MaxPool2d if pool == "max" else nn.AvgPool2d
        k = kernel_size; c1, c2, c3 = n_filters, 2 * n_filters, 4 * n_filters
        self.conv1 = HybridConv(in_ch, c1, k, use_handcrafted); self.act1 = Act(); self.pool1 = Pool(2)
        self.conv2 = nn.Conv2d(c1, c2, k, padding=k // 2);      self.act2 = Act(); self.pool2 = Pool(2)
        self.conv3 = nn.Conv2d(c2, c3, k, padding=k // 2);      self.act3 = Act(); self.pool3 = Pool(2)
        self.flatten = nn.Flatten()
        self.fc_latent = nn.Linear(c3 * (img_size // 8) ** 2, latent_dim)
        self.act_latent = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(latent_dim, n_classes)
    def features(self, x):                       # -> latent vector
        x = self.pool1(self.act1(self.conv1(x)))
        x = self.pool2(self.act2(self.conv2(x)))
        x = self.pool3(self.act3(self.conv3(x)))
        return self.act_latent(self.fc_latent(self.flatten(x)))
    def forward(self, x):
        return self.classifier(self.dropout(self.features(x)))

def layer_complexity_table(model, x):
    """Output shape, parameters and MACs of every conv / pool / dense layer (via forward hooks)."""
    rows, hooks = [], []
    def hook(name):
        def fn(m, i, o):
            inp = i[0]; n_par = sum(p.numel() for p in m.parameters())
            if isinstance(m, (HybridConv, nn.Conv2d)):
                k = m.learn.kernel_size[0] if isinstance(m, HybridConv) else m.kernel_size[0]
                macs = o.shape[2] * o.shape[3] * o.shape[1] * inp.shape[1] * k * k
            elif isinstance(m, nn.Linear):
                macs = m.in_features * m.out_features
            else:
                macs = 0
            rows.append({"layer": name, "type": type(m).__name__, "input": tuple(inp.shape[1:]),
                         "output": tuple(o.shape[1:]), "trainable params": n_par, "MACs": macs})
        return fn
    for name, m in model.named_children():
        hooks.append(m.register_forward_hook(hook(name)))
    model.eval()
    with torch.no_grad(): model(x)
    for h in hooks: h.remove()
    return pd.DataFrame(rows)

def receptive_field(layers):
    """layers: list of (name, kernel, stride). Returns RF (in input pixels) after each layer."""
    rf, jump, out = 1, 1, []
    for name, k, s in layers:
        rf += (k - 1) * jump; jump *= s
        out.append({"layer": name, "receptive field (px)": rf, "jump/stride w.r.t input": jump})
    return pd.DataFrame(out)

set_seed()
model_untrained = MedCNN().to(device)
tbl = layer_complexity_table(model_untrained, x_sample.to(device))
display(tbl)
print(f"Total trainable params: {sum(p.numel() for p in model_untrained.parameters() if p.requires_grad):,} | "
      f"Total MACs per image: {tbl['MACs'].sum():,}")
display(receptive_field([("conv1", 3, 1), ("pool1", 2, 2), ("conv2", 3, 1), ("pool2", 2, 2), ("conv3", 3, 1), ("pool3", 2, 2)]))

## A5. Training the CNN (classification: Normal vs Pneumonia)
* Loss: cross-entropy **weighted by inverse class frequency** (the dataset is imbalanced, so plain accuracy is misleading).
* Model selection: checkpoint with the best **validation ROC-AUC**; the **test set is used only once** for final reporting.

In [ ]:
OPTIMIZERS = {
    "adam":    lambda p, lr: torch.optim.Adam(p, lr=lr),
    "sgd":     lambda p, lr: torch.optim.SGD(p, lr=lr, momentum=0.9),
    "rmsprop": lambda p, lr: torch.optim.RMSprop(p, lr=lr),
}
BASE_CFG = dict(lr=1e-3, kernel_size=3, n_filters=16, optimizer="adam", batch_size=64,
                pool="max", act="relu", dropout=0.3, latent_dim=LATENT_DIM)

@torch.no_grad()
def evaluate(model, X, y, bs=256):
    model.eval(); probs = []
    for i in range(0, len(X), bs):
        probs.append(model(X[i:i + bs].to(device)).softmax(1)[:, 1].cpu())
    p = torch.cat(probs).numpy(); yt = y.numpy(); pred = (p >= 0.5).astype(int)
    auc = roc_auc_score(yt, p) if len(np.unique(yt)) > 1 else float("nan")
    return {"acc": accuracy_score(yt, pred), "auc": auc, "f1": f1_score(yt, pred), "prob": p, "pred": pred}

def train_model(cfg, Xtr, ytr, Xva, yva, epochs, seed=SEED, verbose=True):
    set_seed(seed)
    mcfg = {k: cfg[k] for k in ["kernel_size", "n_filters", "pool", "act", "dropout", "latent_dim"]}
    model = MedCNN(**mcfg).to(device)
    opt = OPTIMIZERS[cfg["optimizer"]](model.parameters(), cfg["lr"])
    counts = torch.bincount(ytr, minlength=2).float()
    crit = nn.CrossEntropyLoss(weight=(counts.sum() / (2 * counts)).to(device))
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=cfg["batch_size"], shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    hist, best_auc, best_state = [], -1, None
    t0 = time.perf_counter()
    for ep in range(1, epochs + 1):
        model.train(); tot, correct, n = 0.0, 0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); out = model(xb); loss = crit(out, yb); loss.backward(); opt.step()
            tot += loss.item() * len(yb); correct += (out.argmax(1) == yb).sum().item(); n += len(yb)
        va = evaluate(model, Xva, yva)
        hist.append({"epoch": ep, "train_loss": tot / n, "train_acc": correct / n, "val_acc": va["acc"], "val_auc": va["auc"]})
        if va["auc"] > best_auc:
            best_auc, best_state = va["auc"], copy.deepcopy(model.state_dict())
        if verbose:
            print(f"epoch {ep:2d} | loss {tot/n:.4f} | train acc {correct/n:.3f} | val acc {va['acc']:.3f} | val AUC {va['auc']:.3f}")
    model.load_state_dict(best_state)
    return model, pd.DataFrame(hist), time.perf_counter() - t0

model, hist, train_time = train_model(BASE_CFG, Xtr, ytr, Xva, yva, EPOCHS)
test_res = evaluate(model, Xte, yte)
print(f"\nTraining time {train_time:.1f}s | TEST  acc={test_res['acc']:.3f}  AUC={test_res['auc']:.3f}  F1={test_res['f1']:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(hist.epoch, hist.train_loss, "o-"); axes[0].set_title("Training loss"); axes[0].set_xlabel("epoch")
axes[1].plot(hist.epoch, hist.train_acc, "o-", label="train acc"); axes[1].plot(hist.epoch, hist.val_acc, "s-", label="val acc")
axes[1].plot(hist.epoch, hist.val_auc, "^-", label="val AUC"); axes[1].legend(); axes[1].set_title("Accuracy / AUC"); axes[1].set_xlabel("epoch")
ConfusionMatrixDisplay(confusion_matrix(yte.numpy(), test_res["pred"]), display_labels=CLASS_NAMES).plot(ax=axes[2], colorbar=False)
axes[2].set_title("Test confusion matrix"); plt.tight_layout(); plt.show()

## A6. Extract & visualise: feature maps after every CNN layer and the latent vector
Forward hooks capture the output of each layer for the selected X-ray. Channels 0–7 of Conv1 are the **fixed user-defined filters** (labelled), the rest are **learned** filters.

In [ ]:
LAYERS = ["conv1", "act1", "pool1", "conv2", "act2", "pool2", "conv3", "act3", "pool3", "flatten", "act_latent"]
fx = FeatureExtractor(model, LAYERS)
feats, logits = fx(x_sample.to(device)); fx.remove()
prob = logits.softmax(1)[0].cpu().numpy()

plt.figure(figsize=(4, 4)); plt.imshow(sample_raw, cmap="gray"); plt.axis("off")
plt.title(f"Original input | true: {CLASS_NAMES[y_train[SAMPLE_IDX]]}\npred: {CLASS_NAMES[prob.argmax()]} ({prob.max():.2f})"); plt.show()

print(pd.DataFrame([{"layer": n, "output shape": tuple(feats[n].shape[1:]), "values": feats[n].numel()} for n in LAYERS]).to_string(index=False))

c1_names = [f"{n} (fixed)" for n in list(USER_FILTERS)[:model.conv1.n_fixed]]
show_feature_maps(feats["conv1"][0], "Conv1 output (after convolution)", names=c1_names, cmap="gray")
show_feature_maps(feats["act1"][0],  "Conv1 → ReLU", names=c1_names)
show_feature_maps(feats["pool1"][0], "Pool1 output (MaxPool 2×2)", names=c1_names)
show_feature_maps(feats["conv2"][0], "Conv2 output")
show_feature_maps(feats["act2"][0],  "Conv2 → ReLU")
show_feature_maps(feats["pool2"][0], "Pool2 output")
show_feature_maps(feats["conv3"][0], "Conv3 output")
show_feature_maps(feats["act3"][0],  "Conv3 → ReLU")
show_feature_maps(feats["pool3"][0], "Pool3 output")

In [ ]:
# Mean activation per stage (how spatial detail is progressively abstracted)
stage = ["act1", "pool1", "act2", "pool2", "act3", "pool3"]
show_images([feats[s][0].mean(0).numpy() for s in stage], [f"{s} mean {tuple(feats[s].shape[2:])}" for s in stage],
            ncols=6, cmap="inferno", size=2.6, suptitle="Channel-averaged activation at each stage")

flat = feats["flatten"][0].numpy(); latent = feats["act_latent"][0].numpy()
fig, axes = plt.subplots(3, 1, figsize=(16, 9))
axes[0].plot(flat, lw=0.5); axes[0].set_title(f"Flattened feature vector (length {flat.size} = {tuple(feats['pool3'].shape[1:])})")
axes[1].bar(range(latent.size), latent, color="teal"); axes[1].set_title(f"Final LATENT vector (dense layer, dim {latent.size}) — "
                                                                         f"{(latent == 0).mean()*100:.0f}% zeros after ReLU")
axes[2].imshow(latent.reshape(8, -1), cmap="viridis", aspect="auto"); axes[2].set_title("Latent vector reshaped as heat-map")
plt.tight_layout(); plt.show()
print("Latent vector (first 20 values):", np.round(latent[:20], 3))

In [ ]:
# Untrained vs trained: same image, same architecture
fx0 = FeatureExtractor(model_untrained, ["act1", "act3", "act_latent"]); f0, _ = fx0(x_sample.to(device)); fx0.remove()
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for r, (lab, f) in enumerate([("Untrained", f0), ("Trained", feats)]):
    axes[r, 0].imshow(f["act1"][0, -1], cmap="viridis"); axes[r, 0].set_title(f"{lab}: a learnable conv1 channel")
    axes[r, 1].imshow(f["act3"][0].mean(0), cmap="inferno"); axes[r, 1].set_title(f"{lab}: conv3 mean activation")
    axes[r, 2].bar(range(LATENT_DIM), f["act_latent"][0], color="teal"); axes[r, 2].set_title(f"{lab}: latent vector")
for a in axes[:, :2].ravel(): a.axis("off")
plt.tight_layout(); plt.show()

## A7. Latent-space analysis (is the latent vector *meaningful*?)
A latent representation is useful if images of the same class cluster together. We check this three ways:
* **PCA** and **t-SNE** projections of test-set latent vectors.
* **Linear probe**: a logistic-regression classifier trained on (a) raw pixels, (b) untrained-CNN latents, (c) trained-CNN latents. Higher probe AUC → more linearly separable (better) representation.

In [ ]:
@torch.no_grad()
def extract_latents(m, X, bs=256):
    m.eval(); return torch.cat([m.features(X[i:i + bs].to(device)).cpu() for i in range(0, len(X), bs)]).numpy()

n_probe = min(len(Xtr), 3000)
Z_tr_tr, Z_te_tr = extract_latents(model, Xtr[:n_probe]), extract_latents(model, Xte)
Z_tr_un, Z_te_un = extract_latents(model_untrained, Xtr[:n_probe]), extract_latents(model_untrained, Xte)
P_tr, P_te = Xtr[:n_probe].flatten(1).numpy(), Xte.flatten(1).numpy()

probe_rows = []
for name, (A, B) in {"Raw pixels": (P_tr, P_te), "Untrained CNN latent": (Z_tr_un, Z_te_un), "Trained CNN latent": (Z_tr_tr, Z_te_tr)}.items():
    sc = StandardScaler().fit(A)
    clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(sc.transform(A), y_train[:n_probe])
    p = clf.predict_proba(sc.transform(B))[:, 1]
    probe_rows.append({"representation": name, "dim": A.shape[1], "probe test acc": accuracy_score(y_test, p >= 0.5),
                       "probe test AUC": roc_auc_score(y_test, p)})
probe_df = pd.DataFrame(probe_rows); display(probe_df.round(4))

n_vis = min(len(Z_te_tr), 1000)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, (title, Z) in zip(axes, [("PCA — untrained latent", PCA(2).fit_transform(Z_te_un[:n_vis])),
                                 ("PCA — trained latent", PCA(2).fit_transform(Z_te_tr[:n_vis])),
                                 ("t-SNE — trained latent", TSNE(2, random_state=SEED, init="pca", perplexity=30).fit_transform(Z_te_tr[:n_vis]))]):
    for c in range(2):
        m_ = y_test[:n_vis] == c
        ax.scatter(Z[m_, 0], Z[m_, 1], s=6, alpha=0.6, label=CLASS_NAMES[c])
    ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()

### Grad-CAM (which regions drive the prediction?)

In [ ]:
show_idx = [int(np.where(y_test == c)[0][k]) for c in range(2) for k in range(2)]
fig, axes = plt.subplots(2, 4, figsize=(15, 7.5))
for j, i in enumerate(show_idx):
    x = Xte[i:i + 1].to(device)
    cam, cls, p = grad_cam(model, x, model.conv3)
    img = X_test_raw[i]
    axes[0, j].imshow(img, cmap="gray"); axes[0, j].set_title(f"true: {CLASS_NAMES[y_test[i]]}"); axes[0, j].axis("off")
    axes[1, j].imshow(sktransform.resize(img, (IMG_SIZE, IMG_SIZE)), cmap="gray"); axes[1, j].imshow(cam, cmap="jet", alpha=0.45)
    axes[1, j].set_title(f"Grad-CAM → {CLASS_NAMES[cls]} ({p:.2f})"); axes[1, j].axis("off")
plt.suptitle("Grad-CAM on conv3 of MedCNN", fontweight="bold"); plt.tight_layout(); plt.show()

## A8. Effect of hyperparameters on the CNN classifier
**Design:** one-factor-at-a-time around the baseline configuration (`BASE_CFG`). Each run uses the same training subset, epochs and seed(s); selection by validation AUC; test metrics reported for the selected checkpoint.

**Limitation (be honest when writing conclusions):** with a single seed and few epochs, differences of ~1–2 % may be noise. Add more seeds in `HP_SEEDS` for error bars. One-factor-at-a-time also ignores interactions between hyperparameters (e.g. learning rate × optimiser).

In [ ]:
HP_GRID = {
    "lr":          [1e-2, 1e-3, 1e-4],
    "kernel_size": [3, 5, 7],
    "n_filters":   [8, 16, 32],
    "optimizer":   ["adam", "sgd", "rmsprop"],
    "batch_size":  [32, 64, 128],
    "pool":        ["max", "avg"],
    "act":         ["relu", "leaky_relu", "tanh"],
    "dropout":     [0.0, 0.3, 0.5],
    "latent_dim":  [32, 128, 256],
}
rng = np.random.default_rng(SEED)
sub = torch.from_numpy(rng.choice(len(Xtr), min(HP_TRAIN_SUBSET, len(Xtr)), replace=False))
Xs, ys = Xtr[sub], ytr[sub]

cache, results = {}, []
for factor, values in HP_GRID.items():
    for v in values:
        cfg = {**BASE_CFG, factor: v}
        key = tuple(sorted(cfg.items()))
        if key not in cache:
            runs = []
            for s in HP_SEEDS:
                m, h, t = train_model(cfg, Xs, ys, Xva, yva, HP_EPOCHS, seed=s, verbose=False)
                te = evaluate(m, Xte, yte)
                runs.append({"val_auc": h.val_auc.max(), "test_acc": te["acc"], "test_auc": te["auc"], "test_f1": te["f1"],
                             "time_s": t, "params": sum(p.numel() for p in m.parameters() if p.requires_grad)})
            cache[key] = pd.DataFrame(runs)
        r = cache[key]
        results.append({"factor": factor, "value": str(v), **r.mean().to_dict(),
                        "test_auc_std": r.test_auc.std() if len(r) > 1 else 0.0})
        print(f"{factor:12s}={str(v):10s} val AUC {r.val_auc.mean():.3f} | test acc {r.test_acc.mean():.3f} "
              f"| test AUC {r.test_auc.mean():.3f} | F1 {r.test_f1.mean():.3f} | {r.time_s.mean():.1f}s")
hp_df = pd.DataFrame(results)

In [ ]:
display(hp_df.round(4))
factors = list(HP_GRID); ncols = 3; nrows = int(np.ceil(len(factors) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
for ax, f in zip(axes.ravel(), factors):
    d = hp_df[hp_df.factor == f]; xpos = np.arange(len(d))
    ax.bar(xpos - 0.2, d.test_acc, 0.4, label="test acc")
    ax.bar(xpos + 0.2, d.test_auc, 0.4, yerr=d.test_auc_std, label="test AUC")
    ax.set_xticks(xpos); ax.set_xticklabels(d.value); ax.set_ylim(0.4, 1.0); ax.set_title(f"Effect of {f}")
    for x_, t_ in zip(xpos, d.time_s): ax.text(x_, 0.42, f"{t_:.0f}s", ha="center", fontsize=8)
axes.ravel()[0].legend()
for ax in axes.ravel()[len(factors):]: ax.axis("off")
plt.suptitle("Hyperparameter study (bars: metrics, text: training time)", fontweight="bold"); plt.tight_layout(); plt.show()

# Sensitivity: how much does each factor move test AUC?
sens = hp_df.groupby("factor").test_auc.agg(lambda s: s.max() - s.min()).sort_values(ascending=False)
print("Sensitivity (max − min test AUC across the values tried):"); print(sens.round(4).to_string())
best = hp_df.loc[hp_df.groupby("factor").val_auc.idxmax(), ["factor", "value", "val_auc", "test_auc"]]
print("\nBest value per factor (chosen by VALIDATION AUC, not test):"); print(best.round(4).to_string(index=False))

---
# PART B — RGB Image: Hierarchical Latent Feature Extraction

## B1. Input RGB image
Default: `skimage.data.chelsea()` (a photo of a cat, bundled with scikit-image, no download). Set `RGB_IMAGE_PATH` in the config to use your own image.

In [ ]:
if RGB_IMAGE_PATH:
    rgb = skio.imread(RGB_IMAGE_PATH)
    rgb = color.gray2rgb(rgb) if rgb.ndim == 2 else rgb[..., :3]
    rgb = (rgb * 255).astype(np.uint8) if rgb.dtype != np.uint8 else rgb
else:
    rgb = skdata.chelsea()
print("RGB image:", rgb.shape, rgb.dtype, "| range", rgb.min(), "-", rgb.max())

fig, axes = plt.subplots(1, 5, figsize=(20, 3.8))
axes[0].imshow(rgb); axes[0].set_title("Input RGB image")
for c, (n, cm) in enumerate([("Red", "Reds"), ("Green", "Greens"), ("Blue", "Blues")]):
    axes[c + 1].imshow(rgb[..., c], cmap=cm); axes[c + 1].set_title(f"{n} channel")
for c, col in enumerate("rgb"):
    axes[4].hist(rgb[..., c].ravel(), bins=64, color=col, alpha=0.5)
axes[4].set_title("Per-channel histogram")
for a in axes[:4]: a.axis("off")
plt.tight_layout(); plt.show()

## B2. RGB pre-processing
Resize to 224×224 → scale to [0,1] → per-channel normalisation with ImageNet mean/std (required because the pretrained network in B5 was trained with these statistics) → tensor `[1, 3, 224, 224]`. Manual and in-built versions compared.

In [ ]:
IMNET_MEAN, IMNET_STD = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])

# user-defined
rgb_resized = sktransform.resize(rgb, (RGB_SIZE, RGB_SIZE), anti_aliasing=True)      # float [0,1], HWC
rgb_norm = (rgb_resized - IMNET_MEAN) / IMNET_STD
x_rgb_manual = torch.from_numpy(rgb_norm.transpose(2, 0, 1)).float().unsqueeze(0)       # HWC -> CHW -> NCHW

# in-built
rgb_tf = transforms.Compose([transforms.ToPILImage(), transforms.Resize((RGB_SIZE, RGB_SIZE), antialias=True),
                             transforms.ToTensor(), transforms.Normalize(IMNET_MEAN.tolist(), IMNET_STD.tolist())])
x_rgb = rgb_tf(rgb).unsqueeze(0)
print("manual:", tuple(x_rgb_manual.shape), "| in-built:", tuple(x_rgb.shape),
      "| mean |diff| =", round(float((x_rgb - x_rgb_manual).abs().mean()), 4))
print("Per-channel mean after normalisation:", x_rgb.mean(dim=(0, 2, 3)).numpy().round(3),
      "| std:", x_rgb.std(dim=(0, 2, 3)).numpy().round(3))

def denorm(t):   # tensor [3,H,W] -> displayable HWC
    return np.clip(t.numpy().transpose(1, 2, 0) * IMNET_STD + IMNET_MEAN, 0, 1)
show_images([rgb, rgb_resized, np.clip((rgb_norm - rgb_norm.min()) / (rgb_norm.max() - rgb_norm.min()), 0, 1), denorm(x_rgb[0])],
            ["original", f"resized {RGB_SIZE}×{RGB_SIZE}", "normalised (rescaled for display)", "tensor → de-normalised check"], ncols=4, size=3.4)

## B3. Hand-designed (user-defined + in-built) filters for each feature level
| Level | User-defined | In-built |
|---|---|---|
| **Edge** | Sobel-X/Y, Laplacian via `conv2d_manual`; RGB 3-D kernel via `conv2d_manual_multichannel` | Canny (`skimage.feature.canny`) |
| **Texture** | Gabor bank from its formula (4 orientations × 2 wavelengths) | Local Binary Patterns (`skimage.feature.local_binary_pattern`) |
| **Shape** | Morphological gradient (dilation − erosion) | HOG, Otsu threshold + contours |
| **Semantic** | — (cannot be hand-designed; this is exactly why CNNs are *learned*) | Pretrained CNN (B5) |

In [ ]:
gray = color.rgb2gray(rgb_resized)

# ---------------- EDGE ----------------
gx = conv2d_manual_vec(gray, USER_FILTERS["Sobel-X"], padding=1)
gy = conv2d_manual_vec(gray, USER_FILTERS["Sobel-Y"], padding=1)
mag, ang = np.hypot(gx, gy), np.arctan2(gy, gx)
lap = conv2d_manual_vec(gray, USER_FILTERS["Laplacian"], padding=1)
canny = feature.canny(gray, sigma=2)

# multi-channel (RGB) 3-D kernel: luminance-weighted Sobel-X, manual vs F.conv2d
k3 = np.stack([w_ * USER_FILTERS["Sobel-X"] for w_ in [0.299, 0.587, 0.114]])            # [3,3,3]
rgb_chw = rgb_resized.transpose(2, 0, 1)
m_rgb = conv2d_manual_multichannel(rgb_chw, k3, padding=1)
b_rgb = F.conv2d(torch.from_numpy(rgb_chw).float()[None], torch.from_numpy(k3).float()[None], padding=1)[0, 0].numpy()
print(f"RGB 3-D kernel: max |manual - F.conv2d| = {np.abs(m_rgb - b_rgb).max():.2e}")

show_images([gx, gy, mag, ang, lap, m_rgb, canny],
            ["Sobel-X (manual)", "Sobel-Y (manual)", "Gradient magnitude", "Gradient orientation",
             "Laplacian (manual)", "RGB 3×3×3 Sobel (manual)", "Canny (in-built)"],
            ncols=7, size=2.5, suptitle="EDGE features")

In [ ]:
# ---------------- TEXTURE ----------------
thetas = [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]; lambdas = [6.0, 12.0]
gabor_bank = np.stack([gabor_kernel_manual(15, sigma=0.56 * l, theta=t, lambd=l) for l in lambdas for t in thetas])
g_names = [f"θ={int(np.degrees(t))}° λ={int(l)}" for l in lambdas for t in thetas]
gab_resp = F.conv2d(torch.from_numpy(gray).float()[None, None], torch.from_numpy(gabor_bank).float()[:, None], padding=7)[0].abs().numpy()
show_images(list(gabor_bank), g_names, ncols=8, cmap="bwr", size=1.9, suptitle="User-defined Gabor kernels")
show_images(list(gab_resp), g_names, ncols=8, cmap="magma", size=1.9, suptitle="TEXTURE: |Gabor response| (applied with in-built F.conv2d)")

g8 = (gray * 255).astype(np.uint8)
lbp = feature.local_binary_pattern(g8, P=8, R=1, method="uniform")
local_var = filters.gaussian(gray**2, 3) - filters.gaussian(gray, 3) ** 2
sk_gabor, _ = filters.gabor(gray, frequency=1 / 6, theta=0)
fig, axes = plt.subplots(1, 4, figsize=(17, 3.8))
axes[0].imshow(lbp, cmap="gray"); axes[0].set_title("LBP (in-built)")
axes[1].bar(range(10), np.histogram(lbp, bins=10, range=(0, 10))[0]); axes[1].set_title("LBP histogram = texture descriptor")
axes[2].imshow(local_var, cmap="magma"); axes[2].set_title("Local variance (texture energy)")
axes[3].imshow(np.abs(sk_gabor), cmap="magma"); axes[3].set_title("skimage.filters.gabor (in-built)")
for a in [axes[0], axes[2], axes[3]]: a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ---------------- SHAPE ----------------
hog_vec, hog_img = feature.hog(gray, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), visualize=True)
morph_grad = morphology.dilation(gray, morphology.disk(2)) - morphology.erosion(gray, morphology.disk(2))
mask = morphology.closing(gray > filters.threshold_otsu(gray), morphology.disk(3))
contours = measure.find_contours(mask.astype(float), 0.5)
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
axes[0].imshow(morph_grad, cmap="gray"); axes[0].set_title("Morphological gradient (user-defined op)")
axes[1].imshow(np.clip(hog_img / hog_img.max() * 3, 0, 1), cmap="gray"); axes[1].set_title(f"HOG (in-built), {hog_vec.size}-D descriptor")
axes[2].imshow(mask, cmap="gray"); axes[2].set_title("Otsu threshold + closing")
axes[3].imshow(rgb_resized)
for c in sorted(contours, key=len)[-5:]: axes[3].plot(c[:, 1], c[:, 0], lw=2)
axes[3].set_title("Largest contours (shape outline)")
for a in axes: a.axis("off")
plt.suptitle("SHAPE features", fontweight="bold"); plt.tight_layout(); plt.show()

## B4. Custom RGB CNN (RGB-LatentCNN) — full pipeline visualisation
Architecture: 4 × [Conv 3×3 → ReLU → MaxPool 2×2] → Flatten → Dense (latent 256) → classifier.
The first layer is a `HybridConv` holding the user-defined edge filters (applied to all 3 colour channels) + learnable filters.

> **Important caveat:** no RGB training set is specified in the lab, so this network is **untrained** (random weights beyond the fixed filters). Its deeper layers are therefore *random projections*, not learned semantic detectors. It demonstrates the mechanics (shapes, channels, pooling, flattening, latent vector). The **learned** hierarchy is demonstrated with a pretrained network in B5.

In [ ]:
class RGBLatentCNN(nn.Module):
    def __init__(self, widths=(16, 32, 64, 128), latent_dim=256, img_size=RGB_SIZE, n_classes=10):
        super().__init__()
        self.conv1 = HybridConv(3, widths[0], 3, use_handcrafted=True); self.act1 = nn.ReLU(); self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(widths[0], widths[1], 3, padding=1);    self.act2 = nn.ReLU(); self.pool2 = nn.MaxPool2d(2)
        self.conv3 = nn.Conv2d(widths[1], widths[2], 3, padding=1);    self.act3 = nn.ReLU(); self.pool3 = nn.MaxPool2d(2)
        self.conv4 = nn.Conv2d(widths[2], widths[3], 3, padding=1);    self.act4 = nn.ReLU(); self.pool4 = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        self.fc_latent = nn.Linear(widths[3] * (img_size // 16) ** 2, latent_dim); self.act_latent = nn.ReLU()
        self.classifier = nn.Linear(latent_dim, n_classes)
    def forward(self, x):
        for i in range(1, 5):
            x = getattr(self, f"pool{i}")(getattr(self, f"act{i}")(getattr(self, f"conv{i}")(x)))
        return self.classifier(self.act_latent(self.fc_latent(self.flatten(x))))

set_seed()
rgb_cnn = RGBLatentCNN().to(device)
display(layer_complexity_table(rgb_cnn, x_rgb.to(device)))
L = [f"{p}{i}" for i in range(1, 5) for p in ("conv", "act", "pool")] + ["flatten", "act_latent"]
fx = FeatureExtractor(rgb_cnn, L); rf_, _ = fx(x_rgb.to(device)); fx.remove()
names_rgb = [f"{n} (fixed)" for n in list(USER_FILTERS)[:rgb_cnn.conv1.n_fixed]]
for i in range(1, 5):
    show_feature_maps(rf_[f"conv{i}"][0], f"RGB-CNN Conv{i} output channels", names=names_rgb if i == 1 else None, cmap="gray")
    show_feature_maps(rf_[f"pool{i}"][0], f"RGB-CNN Pool{i} output (after ReLU + MaxPool)", names=names_rgb if i == 1 else None, max_maps=8)

In [ ]:
flat_rgb, lat_rgb = rf_["flatten"][0].numpy(), rf_["act_latent"][0].numpy()
fig, axes = plt.subplots(2, 1, figsize=(16, 6))
axes[0].plot(flat_rgb, lw=0.3); axes[0].set_title(f"RGB-CNN flattened representation (length {flat_rgb.size})")
axes[1].bar(range(lat_rgb.size), lat_rgb, color="darkorange"); axes[1].set_title(f"RGB-CNN latent vector before classifier (dim {lat_rgb.size})")
plt.tight_layout(); plt.show()

## B5. Pretrained CNN (ResNet-18, ImageNet) — learned hierarchy from edges to semantics
ResNet-18 has 20 convolution layers grouped into a stem (`conv1` 7×7) and 4 stages (`layer1`…`layer4`). We hook **every** convolution layer, the pooling layers (`maxpool`, global `avgpool`), and the 512-D **latent vector that feeds the final classification layer (`fc`)**.

In [ ]:
try:
    rn_weights = models.ResNet18_Weights.IMAGENET1K_V1
    resnet = models.resnet18(weights=rn_weights); PRETRAINED = True
except Exception as e:
    print("⚠️  Could not download pretrained weights ->", repr(e)[:120]); print("⚠️  Using RANDOM weights: B5/B6 semantic results will be meaningless.")
    rn_weights = models.ResNet18_Weights.IMAGENET1K_V1
    resnet = models.resnet18(weights=None); PRETRAINED = False
IMAGENET_CLASSES = rn_weights.meta["categories"]
resnet = resnet.to(device).eval()

conv_names = [n for n, m in resnet.named_modules() if isinstance(m, nn.Conv2d)]
STAGES = ["relu", "maxpool", "layer1", "layer2", "layer3", "layer4", "avgpool"]
fx = FeatureExtractor(resnet, conv_names + STAGES + ["fc"])
rfeat, rlogits = fx(x_rgb.to(device)); fx.remove()
latent_rgb = torch.flatten(rfeat["avgpool"], 1)[0].numpy()          # the latent vector fed to fc

probs = rlogits.softmax(1)[0].cpu()
top5 = probs.topk(5)
print("Pretrained:", PRETRAINED, "| Top-5 ImageNet predictions:")
for p, i in zip(top5.values, top5.indices): print(f"   {IMAGENET_CLASSES[i]:25s} {p:.3f}")
print(f"\n{len(conv_names)} conv layers hooked:")
print(pd.DataFrame([{"conv layer": n, "output (C,H,W)": tuple(rfeat[n].shape[1:])} for n in conv_names]).to_string(index=False))

In [ ]:
# Learned first-layer filters (compare with the hand-designed Sobel / Gabor kernels above)
w1 = resnet.conv1.weight.detach().cpu().numpy()                      # [64,3,7,7]
w1n = (w1 - w1.min()) / (w1.max() - w1.min())
show_images([w1n[i].transpose(1, 2, 0) for i in range(32)], None, ncols=16, size=1.0,
            suptitle="ResNet-18 learned conv1 filters (7×7×3) — many look like oriented edge / Gabor / colour-blob detectors")

# Output channels after EVERY convolution layer (6 channels each)
n_show = 6
fig, axes = plt.subplots(len(conv_names), n_show + 1, figsize=(1.9 * (n_show + 1), 1.9 * len(conv_names)))
for r, n in enumerate(conv_names):
    f = rfeat[n][0]
    axes[r, 0].imshow(f.mean(0), cmap="inferno"); axes[r, 0].set_ylabel(n, fontsize=7, rotation=0, ha="right")
    axes[r, 0].set_title(f"mean {tuple(f.shape)}", fontsize=7)
    for c in range(n_show):
        axes[r, c + 1].imshow(f[c], cmap="viridis"); axes[r, c + 1].set_title(f"ch {c}", fontsize=7)
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
plt.suptitle("Output channels after each convolution layer of ResNet-18", fontweight="bold", y=1.0); plt.tight_layout(); plt.show()

In [ ]:
# Pooling outputs, flattened representation and latent vector
show_feature_maps(rfeat["relu"][0], "Stem conv1 → BN → ReLU", max_maps=8)
show_feature_maps(rfeat["maxpool"][0], "MaxPool (stem) output", max_maps=8)
show_feature_maps(rfeat["layer4"][0], "layer4 output (highest-level spatial maps, 7×7)", max_maps=16, cmap="inferno")

flat_l4 = torch.flatten(rfeat["layer4"], 1)[0].numpy()
fig, axes = plt.subplots(3, 1, figsize=(16, 9))
axes[0].plot(flat_l4, lw=0.2); axes[0].set_title(f"Flattened layer4 feature maps (512×7×7 = {flat_l4.size})")
axes[1].bar(range(512), latent_rgb, color="purple"); axes[1].set_title("Global-AvgPool output = 512-D LATENT vector before classification layer (fc)")
axes[2].imshow(latent_rgb.reshape(16, 32), cmap="viridis", aspect="auto"); axes[2].set_title("Latent vector as heat-map")
plt.tight_layout(); plt.show()
top_units = np.argsort(latent_rgb)[::-1][:10]
print("Most active latent units:", top_units, "values:", latent_rgb[top_units].round(3))

In [ ]:
# Grad-CAM for the top-1 class -> where the high-level semantic evidence is located
cam, cls, p = grad_cam(resnet, x_rgb.to(device), resnet.layer4)
cam3, _, _ = grad_cam(resnet, x_rgb.to(device), resnet.layer3, class_idx=cls)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
axes[0].imshow(denorm(x_rgb[0])); axes[0].set_title("Input")
axes[1].imshow(denorm(x_rgb[0])); axes[1].imshow(cam3, cmap="jet", alpha=0.45); axes[1].set_title("Grad-CAM layer3")
axes[2].imshow(denorm(x_rgb[0])); axes[2].imshow(cam, cmap="jet", alpha=0.45); axes[2].set_title(f"Grad-CAM layer4 → {IMAGENET_CLASSES[cls]} ({p:.2f})")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

## B6. Comparison — how the CNN extracts different levels of features

### Qualitative: hand-designed filter vs the CNN layer that plays the same role

In [ ]:
levels = [("EDGE", mag, "conv1 → ReLU", rfeat["relu"][0]),
          ("TEXTURE", gab_resp.mean(0), "layer1", rfeat["layer1"][0]),
          ("SHAPE / PARTS", hog_img, "layer2 / layer3", rfeat["layer3"][0]),
          ("SEMANTIC", None, "layer4 + Grad-CAM", rfeat["layer4"][0])]
fig, axes = plt.subplots(4, 5, figsize=(17, 14))
for r, (lvl, hand, lname, f) in enumerate(levels):
    if hand is not None: axes[r, 0].imshow(hand, cmap="gray")
    else: axes[r, 0].text(0.5, 0.5, "no hand-designed\nequivalent", ha="center", va="center", fontsize=11)
    axes[r, 0].set_title(f"{lvl}: hand-designed", fontsize=10)
    top = f.flatten(1).mean(1).argsort(descending=True)[:3]           # 3 most active channels
    for j, c in enumerate(top):
        axes[r, j + 1].imshow(f[c], cmap="viridis"); axes[r, j + 1].set_title(f"{lname} ch{int(c)} {tuple(f.shape[1:])}", fontsize=9)
    if lvl == "SEMANTIC":
        axes[r, 4].imshow(denorm(x_rgb[0])); axes[r, 4].imshow(cam, cmap="jet", alpha=0.45); axes[r, 4].set_title(f"→ '{IMAGENET_CLASSES[cls]}'", fontsize=9)
    else:
        axes[r, 4].imshow(f.mean(0), cmap="inferno"); axes[r, 4].set_title(f"{lname} channel mean", fontsize=9)
for a in axes.ravel(): a.axis("off")
plt.suptitle("Low → high level features: hand-crafted vs learned CNN", fontweight="bold"); plt.tight_layout(); plt.show()

### Quantitative: what changes from shallow to deep layers?
* **Receptive field** — how many input pixels influence one activation.
* **Sparsity** — fraction of zero activations (post-ReLU).
* **Effective rank** — number of principal components needed to explain 95 % of the variance across channels (how many independent "feature types" are active).
* **Invariance** — cosine similarity between features of the clean image and a perturbed image (noise, blur, rotation, flip, brightness, crop). Measured both on **flattened spatial maps** and on **global-average-pooled** channel vectors.
* **Semantic selectivity** — similarity between the cat image and *different* images; if deep features are semantic, unrelated images should become *less* similar at deeper layers.

In [ ]:
def resnet_rf():
    L = [("conv1", 7, 2), ("maxpool", 3, 2)]
    for stage, s in [("layer1", 1), ("layer2", 2), ("layer3", 2), ("layer4", 2)]:
        L += [(stage, 3, s), (stage, 3, 1), (stage, 3, 1), (stage, 3, 1)]     # 2 BasicBlocks × 2 convs (longest path)
    df = receptive_field(L); return df.groupby("layer", sort=False).last()["receptive field (px)"].to_dict()
RF = resnet_rf(); RF["relu"] = RF["conv1"]

def eff_rank(f, thr=0.95):
    X = f.flatten(1).T.numpy()                       # positions × channels
    if X.shape[0] < 2: return np.nan
    ev = PCA().fit(X).explained_variance_ratio_
    return int(np.searchsorted(np.cumsum(ev), thr) + 1)

stage_list = ["relu", "maxpool", "layer1", "layer2", "layer3", "layer4"]
q = []
for s in stage_list:
    f = rfeat[s][0]
    q.append({"stage": s, "channels": f.shape[0], "spatial": f"{f.shape[1]}×{f.shape[2]}", "receptive field (px)": RF[s],
              "sparsity (% zeros)": round(100 * float((f == 0).float().mean()), 1), "effective rank (95%)": eff_rank(f)})
display(pd.DataFrame(q))

In [ ]:
def perturb(img, kind):
    f = img.astype(np.float64) / 255
    if kind == "gaussian noise": f = f + np.random.default_rng(0).normal(0, 0.08, f.shape)
    if kind == "blur":           f = filters.gaussian(f, sigma=2, channel_axis=-1)
    if kind == "rotate 15°":     f = sktransform.rotate(f, 15, mode="reflect")
    if kind == "h-flip":         f = f[:, ::-1]
    if kind == "brightness ×1.3": f = f * 1.3
    if kind == "crop 80%":
        h, w = f.shape[:2]; f = sktransform.resize(f[int(.1 * h):int(.9 * h), int(.1 * w):int(.9 * w)], (h, w))
    return (np.clip(f, 0, 1) * 255).astype(np.uint8)

def stage_feats(img_uint8):
    fx_ = FeatureExtractor(resnet, stage_list + ["avgpool"])
    f, _ = fx_(rgb_tf(np.ascontiguousarray(img_uint8)).unsqueeze(0).to(device)); fx_.remove(); return f

def cos(a, b): a, b = a.flatten(), b.flatten(); return float(F.cosine_similarity(a, b, dim=0))

clean = stage_feats(rgb)
perts = ["gaussian noise", "blur", "rotate 15°", "h-flip", "brightness ×1.3", "crop 80%"]
inv_sp, inv_gap = {}, {}
for p_ in perts:
    fp = stage_feats(perturb(rgb, p_))
    inv_sp[p_]  = [cos(clean[s], fp[s]) for s in stage_list]
    inv_gap[p_] = [cos(clean[s].mean((2, 3)), fp[s].mean((2, 3))) for s in stage_list]

# Semantic selectivity: cat vs other images
others = {"astronaut": skdata.astronaut(), "coffee": skdata.coffee(), "rocket": skdata.rocket()}
sem = {k: [cos(clean[s].mean((2, 3)), stage_feats(v)[s].mean((2, 3))) for s in stage_list] for k, v in others.items()}

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for p_ in perts:
    axes[0].plot(stage_list, inv_sp[p_], "o-", label=p_); axes[1].plot(stage_list, inv_gap[p_], "o-", label=p_)
for k in sem: axes[2].plot(stage_list, sem[k], "o-", label=f"cat vs {k}")
axes[0].set_title("Invariance — flattened spatial maps"); axes[1].set_title("Invariance — global-avg-pooled")
axes[2].set_title("Similarity to DIFFERENT images (lower = more selective)")
for a in axes: a.set_ylabel("cosine similarity"); a.legend(fontsize=7); a.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()
show_images([perturb(rgb, p_) for p_ in perts] + list(others.values()), perts + list(others), ncols=9, size=1.9, suptitle="Perturbed / other images used")

---
# PART C — Summary comparison: grayscale medical CNN vs RGB CNN

In [ ]:
cmp = pd.DataFrame({
    "Grayscale medical (MedCNN, trained)": {
        "input tensor": str(tuple(x_sample.shape)), "channels in": 1, "conv layers": 3,
        "latent dim": LATENT_DIM, "trainable params": f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}",
        "first-layer filters": f"{model.conv1.n_fixed} fixed user-defined + {model.conv1.learn.out_channels} learned",
        "training": f"{EPOCHS} epochs on {DATASET_NAME}", "test AUC": round(test_res["auc"], 3)},
    "RGB (RGB-LatentCNN, untrained)": {
        "input tensor": str(tuple(x_rgb.shape)), "channels in": 3, "conv layers": 4, "latent dim": 256,
        "trainable params": f"{sum(p.numel() for p in rgb_cnn.parameters() if p.requires_grad):,}",
        "first-layer filters": f"{rgb_cnn.conv1.n_fixed} fixed user-defined + {rgb_cnn.conv1.learn.out_channels} random",
        "training": "none (mechanics demo)", "test AUC": "n/a"},
    "RGB (ResNet-18, pretrained)": {
        "input tensor": str(tuple(x_rgb.shape)), "channels in": 3, "conv layers": len(conv_names), "latent dim": 512,
        "trainable params": f"{sum(p.numel() for p in resnet.parameters()):,}", "first-layer filters": "64 learned 7×7×3",
        "training": "ImageNet-1k" if PRETRAINED else "NONE (weights failed to load)",
        "test AUC": f"top-1: {IMAGENET_CLASSES[top5.indices[0]]} ({top5.values[0]:.2f})"},
})
display(cmp)

print("Auto-generated findings from THIS run (verify against the plots before writing your conclusion):")
print(f" • Linear probe AUC — raw pixels {probe_df.iloc[0]['probe test AUC']:.3f}, untrained latent {probe_df.iloc[1]['probe test AUC']:.3f}, "
      f"trained latent {probe_df.iloc[2]['probe test AUC']:.3f}")
print(f" • Most sensitive hyperparameter (range of test AUC): {sens.index[0]} ({sens.iloc[0]:.3f}); least: {sens.index[-1]} ({sens.iloc[-1]:.3f})")
print(f" • ResNet receptive field grows {RF['relu']} px → {RF['layer4']} px; spatial size {rfeat['relu'].shape[-1]} → {rfeat['layer4'].shape[-1]}; "
      f"channels {rfeat['relu'].shape[1]} → {rfeat['layer4'].shape[1]}")
print(f" • Mean cosine similarity to unrelated images: stem {np.mean([v[0] for v in sem.values()]):.3f} vs layer4 {np.mean([v[-1] for v in sem.values()]):.3f}")

## Conclusion (write after checking your own outputs)
Points the results are expected to support — **confirm each one against the numbers/plots above; if your run contradicts a point, report what you actually observed**:

1. **Pre-processing** — resizing reduces computation quadratically (MACs ∝ H·W); z-score normalisation centres the inputs, which stabilises gradient-based training. Manual and in-built pipelines agree up to interpolation differences.
2. **User-defined vs in-built filters** — the from-scratch convolution matches `F.conv2d` to floating-point precision, but is orders of magnitude slower; hand-designed kernels respond to fixed patterns (edges, blobs), while learned kernels adapt to the data.
3. **Feature hierarchy (grayscale)** — Conv1 maps retain fine anatomical edges (ribs, lung borders); after each pooling step the spatial resolution halves and maps become coarser and more abstract; the dense layer compresses 4096 flattened values into a compact latent vector.
4. **Latent quality** — the trained latent vector should be more linearly separable (higher probe AUC, clearer PCA/t-SNE clusters) than raw pixels or an untrained network's latents. Learning, not the architecture alone, makes the latent meaningful.
5. **Hyperparameters** — learning rate and optimiser typically have the largest effect; kernel size / number of filters trade accuracy against parameters and time. Treat small differences with caution (single seed, few epochs).
6. **Feature hierarchy (RGB)** — early layers: edges and colour contrasts (conv1 filters resemble Sobel/Gabor kernels); middle layers: textures and repeated patterns; deeper layers: object parts and shapes with large receptive fields; the final 512-D latent vector encodes class-level semantics (top-5 predictions, Grad-CAM on the object).
7. **Invariance & selectivity** — deeper, pooled features are generally more robust to small perturbations and less similar to unrelated images than shallow features; global pooling adds translation/flip invariance on its own, which is why both spatial and pooled measures are reported.